# Reusable Template – Bank PD Log-Loss & Cost

**Short name:** `Bank_LogLoss_Cost`  
Drop in a 0/1 default book, score one or more frozen scorecards, optionally noise the outcomes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

def bce(f, y):
    f = np.clip(np.asarray(f, dtype=float), 1e-15, 1-1e-15)
    y = np.asarray(y, dtype=float)
    return -(y * np.log(f) + (1 - y) * np.log(1 - f))

def book_cost(X, y, w, b):
    X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=float); w = np.asarray(w, dtype=float)
    z = (w * X + b) if X.ndim == 1 else (X @ w + b)
    return float(np.mean(bce(sigmoid(z), y)))

def ecl(pd, lgd, ead):
    return np.asarray(pd, dtype=float) * lgd * ead



## 1. Load a default book — replace this cell


In [ ]:
raw = np.loadtxt("data/bank_scorecard_2d.csv", delimiter=",", skiprows=1)
X, y = raw[:, :2], raw[:, 2]
print(X.shape, "default rate", y.mean())



## 2. Score candidate scorecards


In [ ]:
cands = [("S1", np.array([1.,1.]), -3.), ("S2", np.array([1.,1.]), -4.), ("coin-flip", np.zeros(X.shape[1]), 0.)]
for name, w, b in cands:
    print(f"{name:12s} J={book_cost(X,y,w,b):.6f}")



## 3. Optional ECL overlay (finance view)


In [ ]:
pd_s1 = sigmoid(X @ np.array([1.,1.]) - 3)
print("mean PD S1", pd_s1.mean(), "mean ECL @ LGD=0.45 EAD=10k", ecl(pd_s1, 0.45, 10000).mean())



## 4. Mini simulation — misfile rate vs J


In [ ]:
w, b = np.array([1.,1.]), -3.
rng = np.random.default_rng(0)
rates, js = [], []
for noise in np.linspace(0, 0.4, 9):
    y2 = y.copy()
    nflip = int(noise * len(y))
    if nflip:
        idx = rng.choice(len(y), nflip, replace=False)
        y2[idx] = 1 - y2[idx]
    rates.append(noise)
    js.append(book_cost(X, y2, w, b))
plt.plot(rates, js, "o-")
plt.xlabel("misfile rate"); plt.ylabel("J")
plt.title("Frozen scorecard vs dirty outcomes")
plt.grid(True, alpha=0.3); plt.show()
